# Building a Simple Neural Network from Scratch with NumPy

In this mini project, I build a simple feedforward neural network using only NumPy.

The goal is to understand the basic internal structure of a neural network:

- Inputs
- Hidden layers
- Nodes
- Weights
- Biases
- Weighted sum
- Activation function
- Forward propagation
- Final prediction

This example does not train the neural network. It only initializes the network and passes input data forward through the layers.

In [119]:
import numpy as np
from pprint import pprint

## 1. Initialize the Network

The first function creates the structure of the neural network.

Each layer contains nodes.  
Each node contains:

- weights
- bias

The number of weights for each node depends on the number of nodes in the previous layer.

For example, if a node receives 2 inputs, it needs 2 weights.

In [120]:
def initialize_network(num_inputs, num_hidden_layers, num_nodes_hidden, num_nodes_output, seed=None):
    """
    Initialize a simple feedforward neural network.

    Parameters:
    ----------
    num_inputs : int
        Number of input features.

    num_hidden_layers : int
        Number of hidden layers in the network.

    num_nodes_hidden : list
        A list containing the number of nodes in each hidden layer.
        Example: [2, 2] means:
        - hidden layer 1 has 2 nodes
        - hidden layer 2 has 2 nodes

    num_nodes_output : int
        Number of nodes in the output layer.

    seed : int, optional
        Random seed to make the results reproducible.

    Returns:
    -------
    network : dict
        A nested dictionary containing layers, nodes, weights, and biases.
    """

    # Validate that the number of hidden layers matches the hidden node list
    if num_hidden_layers != len(num_nodes_hidden):
        raise ValueError("num_hidden_layers must match the length of num_nodes_hidden.")

    # Create a random number generator
    rng = np.random.default_rng(seed)

    # At the beginning, the previous layer is the input layer
    num_nodes_previous = num_inputs

    # Create an empty dictionary to store the network
    network = {}

    # Loop through hidden layers and the output layer
    for layer_index in range(num_hidden_layers + 1):

        # If this is the final layer, name it output
        if layer_index == num_hidden_layers:
            layer_name = "output"
            num_nodes = num_nodes_output

        # Otherwise, it is a hidden layer
        else:
            layer_name = f"layer_{layer_index + 1}"
            num_nodes = num_nodes_hidden[layer_index]

        # Create an empty dictionary for the current layer
        network[layer_name] = {}

        # Create nodes inside the current layer
        for node_index in range(num_nodes):
            node_name = f"node_{node_index + 1}"

            # Each node gets:
            # - weights connected to the previous layer
            # - one bias value
            network[layer_name][node_name] = {
                "weights": np.round(rng.uniform(size=num_nodes_previous), 2),
                "bias": float(np.round(rng.uniform(), 2))
            }

        # The current layer becomes the previous layer for the next loop
        num_nodes_previous = num_nodes

    return network

## 2. Create a Network

Now we create a network with:

- 2 input values
- 2 hidden layers
- 2 nodes in each hidden layer
- 1 output node

In [145]:
network = initialize_network(2,2,[2,2],1, seed=42)
pprint(net1)

{'layer_1': {'node_1': {'bias': 0.86, 'weights': array([0.77, 0.44])},
             'node_2': {'bias': 0.98, 'weights': array([0.7 , 0.09])}},
 'layer_2': {'node_1': {'bias': 0.13, 'weights': array([0.76, 0.79])},
             'node_2': {'bias': 0.93, 'weights': array([0.45, 0.37])}},
 'output': {'node_1': {'bias': 0.44, 'weights': array([0.64, 0.82])}}}


## 3. Compute the Weighted Sum

Each node calculates a weighted sum.

The formula is:

weighted sum = input1 × weight1 + input2 × weight2 + ... + bias

This is the raw value before applying the activation function.

In [147]:
def compute_weighted_sum(inputs, weights, bias):
    """
    Compute the weighted sum for one node.

    Formula:
    weighted_sum = dot(inputs, weights) + bias

    Parameters:
    ----------
    inputs : list or array
        Input values coming into the node.

    weights : list or array
        Weights connected to the input values.

    bias : float
        Bias value of the node.

    Returns:
    -------
    weighted_sum : float
        The calculated weighted sum.
    """
    # converting inputs and weights to NumPy arrays, not mandatory, just a good practice

    inputs = np.array(inputs, dtype=float)
    weights = np.array(weights, dtype=float)

    # Make sure the number of inputs matches the number of weights
    if len(inputs) != len(weights):
        raise ValueError("The number of inputs must match the number of weights.")

    weighted_sum = np.dot(inputs, weights) + bias

    return float(weighted_sum)

## 4. Node Activation Function

After calculating the weighted sum, we apply an activation function.

Here, we use the sigmoid activation function.

The sigmoid function converts any input value into a value between 0 and 1.

In [148]:
def node_activation(weighted_sum):
    """
    Apply the sigmoid activation function.

    Parameters:
    ----------
    weighted_sum : float
        The raw weighted sum calculated by the node.

    Returns:
    -------
    output : float
        Activated node output between 0 and 1.
    """

    output = 1.0 / (1.0 + np.exp(-weighted_sum))

    return float(output)

## 5. Forward Propagation

Forward propagation means passing data through the network from the input layer to the output layer.

The process is:

1. Start with the original input values.
2. Send them to the first hidden layer.
3. Each node calculates:
   - weighted sum
   - activation output
4. The outputs of one layer become the inputs for the next layer.
5. Continue until the output layer produces the final prediction.

In [153]:
def forward_propagate(network, inputs, verbose=True):
    """
    Perform forward propagation through the network.

    Parameters:
    ----------
    network : dict
        The neural network dictionary containing layers, nodes, weights, and biases.

    inputs : list
        Input values to send through the network.

    verbose : bool
        If True, prints detailed calculation steps.

    Returns:
    -------
    network_predictions : list
        Final prediction values from the output layer.
    """

    # The first layer receives the original input values
    layer_inputs = list(inputs)

    if verbose:
        print(f"Initial inputs: {layer_inputs}")

    # Loop through each layer in the network
    for layer_name, layer_data in network.items():

        # This list will store the outputs of all nodes in the current layer
        layer_outputs = []

        if verbose:
            print(f"\nProcessing {layer_name}")

        # Loop through each node in the current layer
        for node_name, node_data in layer_data.items():

            # Get the node's weights and bias
            weights = node_data["weights"]
            bias = node_data["bias"]

            # Step 1: compute the weighted sum
            weighted_sum = compute_weighted_sum(layer_inputs, weights, bias)

            # Step 2: apply activation function
            node_output = node_activation(weighted_sum)

            # Round output for easier reading
            node_output = round(node_output, 4)

            # Store this node's output
            layer_outputs.append(node_output)

            if verbose:
                print(
                    f"{node_name}: "
                    f"weighted_sum = {weighted_sum:.4f}, "
                    f"activated_output = {node_output}"
                )

        # The outputs of this layer become the inputs for the next layer
        layer_inputs = layer_outputs

        if verbose:
            if layer_name != "output":
                print(f"Outputs from {layer_name}: {layer_outputs}")
            else:
                print(f"Final network prediction: {layer_outputs}")

    # After the final layer, layer_outputs contains the output prediction
    network_predictions = layer_outputs

    return network_predictions

## 6. Test the Network

Now we pass input values through the network.

Since the network has 2 input nodes, we provide 2 input values.

In [154]:
inputs = [0.5, 0.8]

predictions = forward_propagate(network, inputs)

print("\nReturned prediction:", predictions)

Initial inputs: [0.5, 0.8]

Processing layer_1
node_1: weighted_sum = 1.5970, activated_output = 0.8316
node_2: weighted_sum = 1.4020, activated_output = 0.8025
Outputs from layer_1: [0.8316, 0.8025]

Processing layer_2
node_1: weighted_sum = 1.3960, activated_output = 0.8015
node_2: weighted_sum = 1.6011, activated_output = 0.8322
Outputs from layer_2: [0.8015, 0.8322]

Processing output
node_1: weighted_sum = 1.6354, activated_output = 0.8369
Final network prediction: [0.8369]

Returned prediction: [0.8369]
